# 09 · Retrieval error analysis

Why retrieval fails: a rule-based taxonomy of every question whose expected document is not ranked first, for B0 (dense), E1 (original hybrid) and E2 (improved hybrid), on the development (pilot v0.1) and held-out (v0.2) sets, plus traces of the four failure patterns named in the Milestone 5 brief. Analysis only: no retrieval setting was changed on the basis of held-out failures.

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

In [ ]:
EA = {ds: pd.read_json(A.RESULTS / f"error_analysis_{ds}.jsonl", lines=True) for ds in ("pilot_v0.1", "heldout_v0.2")}
TAX = json.loads((A.RESULTS / "error_taxonomy_definitions.json").read_text())
display(pd.DataFrame([(k, v["rule"], v["possible_improvement"]) for k, v in TAX.items()],
                     columns=["failure category", "rule (automatic)", "possible improvement"]))

## Misses and their primary category\n\nA question can match several rules; the *primary* category is the first matching rule in the order of the table above. Categories are automatic heuristics, not human judgements.

In [ ]:
rows = []
for ds, df in EA.items():
    miss = df[df["rank"] != 1]
    t = miss.groupby(["method", "primary_category"]).size().unstack(fill_value=0)
    t.insert(0, "misses", miss.groupby("method").size()); t.insert(0, "dataset", ds); rows.append(t.reset_index())
tax = pd.concat(rows, ignore_index=True).fillna(0)
tax.to_csv(A.RESULTS / "error_taxonomy_counts.csv", index=False)
tax

In [ ]:
cats = [c for c in TAX if c in tax.columns]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), sharex=True)
for ax, ds in zip(axes, EA):
    t = tax[tax.dataset == ds].set_index("method").reindex(["B0", "E1", "E2"])[cats].fillna(0)
    left = np.zeros(len(t))
    shades = plt.cm.Blues(np.linspace(0.95, 0.25, len(cats)))
    for c, col in zip(cats, shades):
        ax.barh(t.index, t[c], left=left, color=col, edgecolor=A.SURFACE, linewidth=1.5, label=c, zorder=2)
        left += t[c].to_numpy()
    for y, v in enumerate(left):
        ax.text(v + 0.5, y, f"{int(v)}", va="center", fontsize=9, color=A.INK)
    ax.set_title(f"Misses at rank 1 by primary category - {ds} (n = 100 questions)"); ax.set_xlabel("questions")
    ax.grid(axis="x"); ax.grid(axis="y", visible=False); ax.invert_yaxis()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.16), fontsize=8.5)
fig.tight_layout(); print(A.save(fig, "09_error_taxonomy"))

## Pattern 1 - Korean questions retrieving the admissions guide (모집요강)

In [ ]:
both = pd.concat(EA.values())
ko = both[(both.language == "ko") & (both["rank"] != 1)]
adm = ko[ko.top3.map(lambda t: bool(t) and t[0]["title"].startswith("모집요강"))]
t1 = pd.DataFrame({"Korean misses": ko.groupby("method").size(), "top-1 = 모집요강": adm.groupby("method").size()}).fillna(0).astype(int)
t1["share"] = (t1["top-1 = 모집요강"] / t1["Korean misses"]).round(2); display(t1)
display(Markdown(f"The admissions guide is the largest Korean document; multilingual-E5 scores Korean text as similar to any "
                 f"Korean question, so it wins on script rather than topic (**same-script distractor**). It is top-1 in "
                 f"{len(adm)} of {len(ko)} Korean misses across methods; E2 reduces it to {t1.loc['E2', 'top-1 = 모집요강'] if 'E2' in t1.index else 0}."))

## Patterns 2-4 - traces

In [ ]:
def trace(ds, qid):
    d = EA[ds][EA[ds].question_id == qid]
    return pd.DataFrame([dict(question=r.question_id, method=r.method, rank=r["rank"], category=r.primary_category,
                              deep_dense_rank=r.deep_dense_rank_no_threshold, expected_best_similarity=r.expected_best_dense_similarity,
                              top1_document=(r.top3[0]["title"][:40] if r.top3 else None), top1_section=(r.top3[0]["heading"][:35] if r.top3 else None))
                         for _, r in d.iterrows()])
display(Markdown("**2. English winter-dormitory question vs the Korean notice (pilot S12-en)**")); display(trace("pilot_v0.1", "S12-en"))
display(Markdown("**3. Vietnamese part-time weekly hours (pilot S06-vi)**")); display(trace("pilot_v0.1", "S06-vi"))
display(Markdown("**4. Travel while an extension is pending (held-out H04 - analysis only)**"))
display(pd.concat([trace("heldout_v0.2", f"H04-{l}") for l in A.LANGS]))

In [ ]:
h = EA["heldout_v0.2"]
thr = h[h.method == "B0"]
display(Markdown(f"""**Similarity threshold.** Expected documents whose best dense similarity is below the 0.80 cut-off:
{int((thr.expected_best_dense_similarity < 0.80).sum())} held-out questions; expected document absent from the dense top-20 even without
a threshold: {int(thr.deep_dense_rank_no_threshold.isna().sum())}. The threshold is a minor cause; most misses are ranking errors
among candidates that *are* retrieved.

**Interpretation.** The dominant failure for E2 is a document in the question's own script outranking the expected
document in another language (same-script distractor), followed by keyword ambiguity (generic terms such as "visa
extension" pulling the wrong guide) and dominance of the 164-chunk general guide. These are exactly the cases a
cross-lingual cross-encoder could address - tested in notebook 10.

**Limitations.** Rule-based categories; AI-drafted, unreviewed labels (a "miss" may be a correct document missing from
the labels - see `possible_label_gap`); 20 questions per language."""))